# 🛰️ تقسیم‌بندی معنایی با یادگیری عمیق
## نوت‌بوک آموزشی کامل و جزئی — Semantic Segmentation with Deep Learning

---

**برای چه کسی نوشته شده؟**
این نوت‌بوک برای کسی نوشته شده که:
- پایتون مقدماتی بلد است (حلقه، تابع، لیست)
- می‌خواهد **هوش مصنوعی بینایی** را در عمل یاد بگیرد
- می‌خواهد همزمان **بفهمد، ببیند و اجرا کند**

**در پایان این نوت‌بوک یاد می‌گیری:**
- Dataset ماهواره‌ای چگونه کار می‌کند
- چگونه یک شبکه عصبی عکس را پیکسل‌به‌پیکسل برچسب می‌زند
- معماری CNN ساده چیست و چگونه کار می‌کند
- معماری U-Net چیست و چرا از CNN ساده بهتر است

---
> **💡 طرز استفاده:** هر Markdown سلول را بخوان، سپس سلول کد زیرش را اجرا کن (دکمه **▶**).
> هیچ‌وقت از یک مرحله رد نشو!"


---
## 📚 مفهوم اول: تقسیم‌بندی معنایی (Semantic Segmentation) چیست؟

### داستان از اینجا شروع می‌شود...

تصور کن از یک پهپاد عکس هوایی گرفته شده. می‌خواهیم بدانیم:
**هر پیکسل این عکس به چه چیزی تعلق دارد؟** ساختمان؟ درخت؟ ماشین؟ جاده؟

این کار را **تقسیم‌بندی معنایی** یا Semantic Segmentation می‌نامیم.

### تفاوت با تشخیص شیء (Object Detection):
- **Object Detection**: یک مستطیل دور شیء می‌کشد → "اینجا یک ماشین است"
- **Semantic Segmentation**: به ازای **هر پیکسل** می‌گوید → "این پیکسل ماشین است، آن پیکسل درخت است"

```
ورودی: تصویر هوایی               خروجی: نقشه رنگی
┌─────────────────┐               ┌─────────────────┐
│  (تصویر RGB)    │  → مدل AI →  │  زرد = ماشین    │
│                 │               │  سبز = درخت     │
│                 │               │  آبی = ساختمان  │
└─────────────────┘               └─────────────────┘
```

### چرا مهم است؟
- 🚗 خودروهای خودران (هر پیکسل جاده/پیاده‌رو/ماشین)
- 🏥 پزشکی (هر پیکسل تومور/بافت سالم)
- 🛰️ نقشه‌برداری (ساختمان‌ها، جاده‌ها، مناطق سبز)


---
## 📦 مفهوم دوم: Dataset — داده‌های پوتسدام

### چرا Dataset پوتسدام؟

**پوتسدام** شهری در آلمان است. سازمان ISPRS تصاویر هوایی با وضوح ۵ سانتیمتر
(یعنی هر پیکسل = ۵×۵ سانتیمتر از زمین) از این شهر گرفته و همه پیکسل‌ها را
**دستی** برچسب زده است. این کار سال‌ها وقت گرفته و Dataset بسیار ارزشمندی ساخته.

### ساختار داده: فایل GeoTIFF چیست؟

GeoTIFF یک فرمت تصویر **چند باندی** است. مثل یک عکس رنگی که به جای ۳ لایه
(RGB)، **۶ لایه** دارد:

```
┌────────────────────────────────────────────────────┐
│  باند ۰: Red (قرمز)          → رنگ قرمز عکس      │
│  باند ۱: Green (سبز)          → رنگ سبز عکس      │
│  باند ۲: Blue (آبی)           → رنگ آبی عکس      │
│  باند ۳: Infrared (مادون‌قرمز) → ناپیدا برای چشم  │
│  باند ۴: Elevation (ارتفاع)   → ارتفاع از سطح دریا│
│  باند ۵: Labels (برچسب)       → ← این هدف ماست! │
└────────────────────────────────────────────────────┘
```

### ۶ کلاس برچسب:
| شماره | نام کلاس | رنگ نمایش |
|---|---|---|
| 0 | Impervious surface (سطح نفوذناپذیر: آسفالت، پیاده‌رو) | ⬜ سفید |
| 1 | Building (ساختمان) | 🟦 آبی |
| 2 | Tree (درخت) | 🟩 سبز |
| 3 | Low vegetation (علف، چمن) | 🩵 آبی-سبز |
| 4 | Car (ماشین) | 🟨 زرد |
| 5 | Clutter/Background (پس‌زمینه) | 🟥 قرمز |

> **سوال:** چرا مادون‌قرمز؟ چون گیاهان مادون‌قرمز را زیاد بازتاب می‌دهند.
> با این باند می‌توان درخت را از علف و آسفالت بهتر تشخیص داد!


---
## 📦 گام ۰ — نصب کتابخانه‌ها

### هر کتابخانه چه کار می‌کند؟

- **`rasterio`**: خواندن فایل‌های GeoTIFF (مثل PIL ولی برای داده‌های جغرافیایی)
- **`numpy`**: کار با آرایه‌های عددی چندبعدی (هر تصویر = یک آرایه عددی)
- **`matplotlib`**: رسم نمودار و نمایش تصویر
- **`scikit-learn`**: ابزار ML از جمله KFold split
- **`tensorflow`**: فریم‌ورک یادگیری عمیق — ساخت و آموزش شبکه عصبی

> ⏳ ممکن است چند دقیقه طول بکشد. پیام `Successfully installed` → یعنی اوکی است.


In [ ]:
# نصب تمام کتابخانه‌های لازم
!pip install rasterio numpy matplotlib scikit-learn tensorflow -q

# بعد از نصب، نسخه TensorFlow را چاپ کن تا مطمئن شوی
import tensorflow as tf
print('TensorFlow نصب شد، نسخه:', tf.__version__)
print('همه چیز آماده است!')


---
## ⚙️ گام ۱ — Import کتابخانه‌ها و تنظیمات اولیه

### این سلول چه کار می‌کند؟

هر چیزی که در کل notebook نیاز داریم را یکجا وارد می‌کنیم.

**توضیح هر خط:**
```python
import os           # کار با فایل‌ها و پوشه‌ها (مثل os.path.join)
import random       # انتخاب تصادفی (مثل random.choice)
import json         # ذخیره و خواندن فایل JSON
import numpy as np  # کار با آرایه عددی — np مخفف numpy است
import matplotlib.pyplot as plt  # رسم نمودار و تصویر
import rasterio     # خواندن GeoTIFF
import tensorflow as tf          # یادگیری عمیق
from tensorflow import keras     # API سطح‌بالای TensorFlow
from tensorflow.keras import layers  # لایه‌های شبکه عصبی
```

### SEED چیست؟
`SEED = 42` یعنی عملیات تصادفی قابل تکرار باشند.
اگر SEED یکسان باشد، همه هربار همان نتیجه را می‌گیرند. عدد ۴۲ قراردادی است!

### DATA_DIR چیست؟
مسیر پوشه‌ای که فایل‌های `.tif` (تایل‌های dataset) در آن هستند.
⚠️ اگر dataset را در جای دیگری داری، این مسیر را تغییر بده.


In [ ]:
import os, random, json
import numpy as np
import matplotlib
matplotlib.use('Agg')  # headless mode
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import rasterio
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.model_selection import KFold
from IPython.display import Image, display

# ── Reproducibility seed ────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Global settings ────────────────────────
N_FOLDS     = 5
NUM_CLASSES = 6

def discover_data_dir():
    possible = [
        os.path.join(os.getcwd(), 'Potsdam-GeoTif'),
        os.path.join(os.getcwd(), 'data'),
        os.path.join(os.getcwd(), 'PROJECT', 'Potsdam-GeoTif'),
        os.path.join(os.getcwd(), 'PROJECT', 'data'),
        os.getcwd()
    ]
    for p in [p for p in possible if os.path.exists(p)]:
        try:
            if any(f.endswith('.tif') for f in os.listdir(p)):
                return p
        except:
            continue
    return 'data'

DATA_DIR = discover_data_dir()
print(f"Data directory: {DATA_DIR}")


---
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# مرحله ۱: آماده‌سازی Dataset
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## هدف این مرحله:
۱. پیدا کردن تمام فایل‌های تایل در dataset
۲. خواندن یک تایل و نمایش باندهای مختلف آن
۳. تقسیم فایل‌ها به ۵ گروه (fold) برای ارزیابی عادلانه

## چرا این مرحله مهم است؟

قبل از اینکه بتوانیم مدل AI بسازیم، باید **داده‌ها را بشناسیم**.
یادگیری عمیق بدون شناخت داده مثل پختن غذا بدون دیدن مواد اولیه است!


### ۱.۱ — پیدا کردن فایل‌های GeoTIFF

#### تابع `get_all_tif_files` چه کار می‌کند؟

```python
os.walk(data_dir)
```
این تابع پوشه را **بازدید می‌کند** — مثل اینکه همه پوشه‌ها و زیرپوشه‌ها را
یکی‌یکی باز کنی و فایل‌هایشان را بنویسی.

```python
if f.endswith('.tif')
```
فقط فایل‌هایی که پسوندشان `.tif` است را انتخاب می‌کنیم.

```python
tif_files.append(os.path.join(root, f))
```
مسیر کامل فایل (پوشه + نام فایل) را به لیست اضافه می‌کنیم.

> با dataset کامل: انتظار داریم بیش از ۱۵,۰۰۰ فایل پیدا شود.
> الان با یک فایل sample کار می‌کنیم — نتایج مثال هستند.


In [ ]:
def get_all_tif_files(data_dir):
    """
    همه فایل‌های .tif را در یک پوشه (و زیرپوشه‌هایش) پیدا می‌کند.
    
    پارامتر: data_dir  → مسیر پوشه‌ای که می‌خواهیم جستجو کنیم
    خروجی : لیستی از مسیرهای کامل فایل‌های .tif
    """
    tif_files = []  # لیست خالی برای نگه داشتن مسیر فایل‌ها
    
    # os.walk مثل باز کردن همه پوشه‌ها است
    # root = مسیر پوشه فعلی
    # dirs = لیست زیرپوشه‌ها
    # files = لیست فایل‌ها
    for root, dirs, files in os.walk(data_dir):
        for f in files:
            if f.endswith('.tif'):  # فقط فایل‌های GeoTIFF
                full_path = os.path.join(root, f)  # مسیر کامل
                tif_files.append(full_path)
    
    return tif_files

# اجرای تابع
all_files = get_all_tif_files(DATA_DIR)

print(f'تعداد فایل‌های GeoTIFF پیدا شده: {len(all_files)}')
print('\nفایل‌های پیدا شده:')
for f in all_files:
    print('  •', os.path.basename(f))  # فقط نام فایل (بدون مسیر)


### ۱.۲ — خواندن و درک یک تایل

#### تابع `rasterio.open` چیست؟

`rasterio` مثل `PIL.Image.open` است ولی برای تصاویر جغرافیایی با چند باند.

```python
with rasterio.open(file_path) as src:
    data = src.read()  # خواندن همه باندها → آرایه numpy به شکل (6, H, W)
```

شکل داده: `(6, 224, 224)` یعنی:
- **۶**: تعداد باندها
- **224**: ارتفاع تصویر (پیکسل)  
- **224**: عرض تصویر (پیکسل)

#### تابع `normalize_band` چیست و چرا نیاز داریم؟

تصور کن ارزش‌های ارتفاع: ۱۰۰ تا ۵۰۰ متر باشد.
`matplotlib` برای نمایش تصویر عدد ۰-۱ یا ۰-۲۵۵ می‌خواهد.
نرمال‌سازی این عدد‌ها را به بازه ۰ تا ۱ تبدیل می‌کند:

```
نرمال = (عدد - حداقل) / (حداکثر - حداقل)
مثال: (300 - 100) / (500 - 100) = 200/400 = 0.5
```

#### تابع `label_to_rgb` چیست؟

باند برچسب یک آرایه از اعداد ۰-۵ است (هر عدد یک کلاس).
برای نمایش، هر عدد را به یک رنگ تبدیل می‌کنیم:
- ۰ → سفید، ۱ → آبی، ۲ → سبز، ...


In [ ]:
# ── توابع کمکی ──────────────────────────────────────────────────────

def normalize_band(band):
    """
    یک باند عددی را به بازه [0, 1] نرمال می‌کند.
    
    چرا؟ برای نمایش صحیح تصویر در matplotlib
    (matplotlib برای float تصویر: باید 0.0 تا 1.0 باشد)
    
    فرمول: normalized = (x - min) / (max - min)
    """
    b_min = band.min()  # حداقل مقدار
    b_max = band.max()  # حداکثر مقدار
    if b_max == b_min:  # اگر همه مقادیر یکسان بودند (از تقسیم بر صفر جلوگیری)
        return np.zeros_like(band, dtype=np.float32)
    return (band - b_min).astype(np.float32) / (b_max - b_min)


def label_to_rgb(label_band, colors):
    """
    باند برچسب (اعداد 0-5) را به تصویر رنگی تبدیل می‌کند.
    
    ورودی: label_band → آرایه (H, W) از اعداد 0-5
           colors → لیست رنگ‌ها، هر رنگ [R, G, B]
    خروجی: rgb → آرایه (H, W, 3) قابل نمایش
    """
    h, w = label_band.shape          # ارتفاع و عرض تصویر
    rgb = np.zeros((h, w, 3), dtype=np.uint8)  # تصویر رنگی خالی (سیاه)
    
    for class_idx, color in enumerate(colors):
        # پیدا کردن تمام پیکسل‌هایی که به این کلاس تعلق دارند
        mask = (label_band == class_idx)
        # اعمال رنگ کلاس به آن پیکسل‌ها
        rgb[mask] = color
    
    return rgb


# ── خواندن یک تایل نمونه ────────────────────────────────────────────

# انتخاب فایل نمونه
EXCLUDED = '0000000224-0000042784.tif'  # این فایل مثال تکلیف است
candidates = [f for f in all_files if EXCLUDED not in f]
sample_file = random.choice(candidates) if candidates else all_files[0]
print('فایل انتخاب شده:', os.path.basename(sample_file))

# خواندن داده با rasterio
with rasterio.open(sample_file) as src:
    data = src.read()           # شکل: (6, H, W)
    crs  = src.crs              # سیستم مختصات جغرافیایی
    transform = src.transform   # اطلاعات مکانی

print(f'\nشکل داده: {data.shape}')
print(f'  → {data.shape[0]} باند | {data.shape[1]} پیکسل ارتفاع | {data.shape[2]} پیکسل عرض')
print(f'سیستم مختصات: {crs}')
print(f'\nبازه مقادیر هر باند:')
band_names = ['Red','Green','Blue','Infrared','Elevation','Labels']
for i, name in enumerate(band_names):
    print(f'  باند {i} ({name}): {data[i].min()} تا {data[i].max()}')


### ۱.۳ — تصویرسازی سه نمای مختلف

#### چرا سه نمای مختلف؟
- **تصویر RGB**: همان چیزی که چشم انسان می‌بیند
- **باند ارتفاع**: نشان می‌دهد ساختمان‌ها از زمین بلندترند
- **نقشه برچسب**: هدف ما — چیزی که مدل باید یاد بگیرد

#### `np.stack` چیست؟
سه آرایه ۲D را روی هم می‌چیند تا یک آرایه ۳D (تصویر رنگی) بسازد:
```
stack([R(224,224), G(224,224), B(224,224)], axis=-1) → (224,224,3)
```

#### `plt.subplots(1, 3)` چیست؟
یک figure با **۱ ردیف** و **۳ ستون** می‌سازد — سه نمودار کنار هم.

#### `plt.colorbar` چیست?
نوار رنگی کنار تصویر که نشان می‌دهد هر رنگ معادل چه عددی است.
برای باند ارتفاع: رنگ روشن‌تر = ارتفاع بیشتر


In [ ]:
# ── ساخت تصاویر برای نمایش ─────────────────────────────────────────

# جدا کردن باندها
red   = data[0].astype(np.float32)   # باند قرمز
green = data[1].astype(np.float32)   # باند سبز
blue  = data[2].astype(np.float32)   # باند آبی
elev  = data[4].astype(np.float32)   # باند ارتفاع
label = data[5].astype(np.int32)     # باند برچسب (عدد صحیح)

# ساخت تصویر RGB: نرمال‌سازی هر باند و ترکیب آن‌ها
# axis=-1 یعنی باندها را در آخرین بُعد کنار هم بگذار
rgb_img = np.stack([
    normalize_band(red),    # R → اعداد 0.0 تا 1.0
    normalize_band(green),  # G → اعداد 0.0 تا 1.0
    normalize_band(blue),   # B → اعداد 0.0 تا 1.0
], axis=-1)  # نتیجه: (H, W, 3)

# نرمال‌سازی باند ارتفاع
elev_normalized = normalize_band(elev)  # نتیجه: (H, W)

# تبدیل برچسب‌ها به رنگ
label_rgb = label_to_rgb(label, CLASS_COLORS)  # نتیجه: (H, W, 3)

# ساخت legend (توضیح رنگ‌ها)
patches = [
    mpatches.Patch(
        color=[c/255 for c in CLASS_COLORS[i]],  # رنگ به فرمت 0-1
        label=CLASS_NAMES[i]                      # نام کلاس
    )
    for i in range(NUM_CLASSES)
]

# ── رسم سه تصویر کنار هم ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('نمایش تایل نمونه از Dataset پوتسدام', fontsize=14, fontweight='bold')

# تصویر ۱: RGB
axes[0].imshow(rgb_img)                   # نمایش آرایه (H,W,3) به عنوان رنگی
axes[0].set_title('تصویر RGB (باندهای 0,1,2)', fontsize=11)
axes[0].axis('off')                       # پنهان کردن محورها

# تصویر ۲: ارتفاع با colorbar
im = axes[1].imshow(elev_normalized, cmap='terrain')  # cmap رنگ‌بندی را تنظیم می‌کند
axes[1].set_title('باند ارتفاع (Elevation)', fontsize=11)
axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04, label='ارتفاع نرمال')

# تصویر ۳: برچسب‌ها
axes[2].imshow(label_rgb)
axes[2].set_title('نقشه برچسب (Label Map)', fontsize=11)
axes[2].axis('off')
axes[2].legend(handles=patches, loc='lower right', fontsize=7, framealpha=0.9)

plt.tight_layout()
out_vis = os.path.join(DATA_DIR, 'step1_visualization.png')
plt.savefig(out_vis, dpi=150, bbox_inches='tight')
plt.close()
display(Image(out_vis))
print('تصویر ذخیره شد:', out_vis)


### ۱.۴ — آمار توزیع کلاس‌ها

#### چرا این آمار مهم است؟

تصور کن ۹۰٪ پیکسل‌ها «آسفالت» باشند و ۱٪ «ماشین».
اگر مدل همیشه «آسفالت» پیش‌بینی کند، دقت ۹۰٪ دارد ولی **بی‌فایده** است!

این توزیع به ما می‌گوید که آیا dataset **متوازن** است یا نه.
نامتوازن بودن dataset چالشی مهم در یادگیری ماشین است.

#### `(label == i).sum()` چه کار می‌کند؟
- `label == i` → یک ماسک True/False با همان شکل label
- `.sum()` → تعداد True‌ها = تعداد پیکسل‌های کلاس i


In [ ]:
# ── آمار کلاس‌ها ────────────────────────────────────────────────────

print('آمار توزیع کلاس‌ها در تایل نمونه:')
print(f'{"کلاس":<30} {"پیکسل":>12} {"درصد":>8} {"نمودار"}')
print('=' * 70)

total_pixels = label.size  # تعداد کل پیکسل‌ها (H × W)

for i, name in enumerate(CLASS_NAMES):
    count   = (label == i).sum()              # تعداد پیکسل این کلاس
    percent = 100 * count / total_pixels      # درصد این کلاس
    bar     = '|' + '█' * int(percent / 2)   # نمودار متنی
    print(f'{name:<30} {count:>12,} {percent:>7.2f}%  {bar}')

print('=' * 70)
print(f'مجموع پیکسل‌ها: {total_pixels:,}  (= {data.shape[1]} × {data.shape[2]})')

# غالب‌ترین کلاس
dominant = max(range(NUM_CLASSES), key=lambda i: (label==i).sum())
print(f'\nکلاس غالب: «{CLASS_NAMES[dominant]}» با {100*(label==dominant).sum()/total_pixels:.1f}%')


### ۱.۵ — تقسیم‌بندی K-Fold

#### مشکل: چگونه مدل را ارزیابی کنیم؟

اگر از همان داده‌ای که مدل روی آن **آموزش دیده** برای **ارزیابی** استفاده کنیم:
مثل این است که از دانش‌آموزی که سوال‌ها را قبلاً دیده امتحان گرفتیم.
نتیجه خوب می‌شود ولی **هیچ معنایی ندارد!**

#### راه‌حل: K-Fold Cross-Validation

تمام داده را به **K قسمت مساوی** (fold) تقسیم می‌کنیم:

```
Dataset: ████████████████████ (100 فایل)
          ├── Fold 1 ──┤├── Fold 2 ──┤├── Fold 3 ──┤├── Fold 4 ──┤├── Fold 5 ──┤
           Training (60%)        Validation (20%)         Test (20%)
              Fold 1+2+3             Fold 4                Fold 5
```

- **Train (Fold 1+2+3)**: مدل روی اینها یاد می‌گیرد
- **Validation (Fold 4)**: در حین آموزش، عملکرد را چک می‌کنیم (انتخاب بهترین مدل)
- **Test (Fold 5)**: هرگز در آموزش دیده نمی‌شود — ارزیابی نهایی واقعی

#### `KFold(n_splits=5)` چیست؟
`scikit-learn` ابزاری آماده برای این تقسیم‌بندی دارد:
- شاخص‌های (0 تا N-1) را به ۵ گروه مساوی تقسیم می‌کند
- `shuffle=True` → اول ترتیب را به هم می‌زند (برای بی‌طرفی)


In [ ]:
# =====================================================================
# =====================================================================
# =====================================================================
# CONFIGURATION - Adjust paths as needed
# =====================================================================
# Original local path:
# DATA_DIR = r'c:\Users\mina_\OneDrive\Documents\DESING_OF_AI_SYSTEMS\Semantic Segmentation with Deep Learning\PROJECT\Potsdam-GeoTif'

import os
def discover_data_dir():
    possible = [
        os.path.join(os.getcwd(), 'Potsdam-GeoTif'),
        os.path.join(os.getcwd(), 'data'),
        os.path.join(os.getcwd(), 'PROJECT', 'Potsdam-GeoTif'),
        os.path.join(os.getcwd(), 'PROJECT', 'data'),
        os.getcwd()
    ]
    existing = [p for p in possible if os.path.exists(p)]
    for p in existing:
        try:
            if any(f.endswith('.tif') for f in os.listdir(p)):
                return p
        except:
            continue
    return existing[0] if existing else 'data'

DATA_DIR = discover_data_dir()
print(f"Data directory: {DATA_DIR}")
# =====================================================================
# =====================================================================
# Original local path:

import os
    os.path.join(os.getcwd(), 'Potsdam-GeoTif'),
    os.path.join(os.getcwd(), 'data'),
    os.path.join(os.getcwd(), 'PROJECT', 'Potsdam-GeoTif'),
    os.path.join(os.getcwd(), 'PROJECT', 'data'),
    os.path.join(os.path.dirname(os.getcwd()), 'Potsdam-GeoTif'),
    os.path.join(os.path.dirname(os.getcwd()), 'data'),
    os.getcwd()
]
# =====================================================================
# =====================================================================
# Original local path:

import os
    os.path.join(os.getcwd(), 'Potsdam-GeoTif'),
    os.path.join(os.getcwd(), 'data'),
    os.path.join(os.path.dirname(os.getcwd()), 'Potsdam-GeoTif'),
    os.path.join(os.path.dirname(os.getcwd()), 'data'),
    os.getcwd()
]
# =====================================================================

# ── تقسیم‌بندی K-Fold ───────────────────────────────────────────────

# اگر فایل‌های کمی داریم، تکرار می‌کنیم (فقط برای demo)
demo_files = all_files * max(1, (N_FOLDS * 4) // max(len(all_files), 1) + 1)
demo_files = demo_files[:max(len(all_files), N_FOLDS * 4)]

# KFold: تقسیم‌کننده K-Fold از scikit-learn
kf = KFold(
    n_splits=N_FOLDS,   # 5 fold
    shuffle=True,        # ترتیب تصادفی
    random_state=SEED    # برای تکرارپذیری
)

# ساخت folds از طریق KFold
arr   = np.array(demo_files)
folds = []  # لیست 5 fold

# kf.split فقط ایندکس‌ها را برمی‌گرداند (نه خود فایل‌ها)
# _ = ایندکس‌های train (که ما نمی‌خواهیم), fold_idx = ایندکس‌های این fold
for _, fold_idx in kf.split(arr):
    folds.append(arr[fold_idx].tolist())  # فایل‌های این fold

# تخصیص folds به train/val/test
train_files = folds[0] + folds[1] + folds[2]  # Fold 1,2,3 → آموزش
val_files   = folds[3]                          # Fold 4     → اعتبارسنجی
test_files  = folds[4]                          # Fold 5     → تست

print('نتیجه تقسیم‌بندی K-Fold:')
for i, fold in enumerate(folds, 1):
    role = 'Train' if i<=3 else ('Val' if i==4 else 'Test')
    print(f'  Fold {i} ({role}): {len(fold)} فایل')
print(f'\nآموزش   : {len(train_files)} فایل')
print(f'اعتبارسنجی: {len(val_files)} فایل')
print(f'تست     : {len(test_files)} فایل')

# ذخیره در JSON برای استفاده در مراحل بعد
splits = {'train': train_files, 'val': val_files,
          'test': test_files, 'all_folds': folds}
splits_path = os.path.join(DATA_DIR, 'fold_splits.json')
with open(splits_path, 'w') as fp:
    json.dump(splits, fp, indent=2)
# # # print('\nفایل fold_splits.json ذخیره شد')
# # print('\nفایل fold_splits.json ذخیره شد')
# # print('\nفایل fold_splits.json ذخیره شد')
# print('\nفایل fold_splits.json ذخیره شد')
# # print('\nفایل fold_splits.json ذخیره شد')
# print('\nفایل fold_splits.json ذخیره شد')
# print('\nفایل fold_splits.json ذخیره شد')
print('\nفایل fold_splits.json ذخیره شد')


### خلاصه مرحله ۱

**چه یاد گرفتیم؟**
- فایل GeoTIFF چیست و چگونه خوانده می‌شود
- نرمال‌سازی باند برای نمایش
- تبدیل برچسب عددی به تصویر رنگی
- K-Fold Cross-Validation برای ارزیابی عادلانه

**چه فایل‌هایی ساخته شد؟**
- `step1_visualization.png` — تصویرسازی باندها
- `fold_splits.json` — تقسیم‌بندی dataset

**→ حالا به مرحله ۲: ساخت اولین مدل یادگیری عمیق!**


---
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# مرحله ۲: مدل ساده شبکه عصبی کانولوشنی (CNN)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## مفهوم اصلی: شبکه عصبی کانولوشنی (CNN) چیست؟

### قبل از CNN: شبکه‌های عصبی ساده

در یک شبکه عصبی ساده (Fully Connected):
- تصویر 224×224 داریم → ۵۰,۱۷۶ پیکسل
- اگر لایه اول ۱۰۰۰ نرون داشته باشد: **۵۰ میلیون وزن** برای **یک لایه**!
- حافظه زیاد، کُند، و overfit می‌کند

### راه‌حل: کانولوشن (Convolution)

**کانولوشن** یعنی یک فیلتر کوچک (مثلاً 3×3) را روی تمام تصویر **اسلاید** کنیم:

```
تصویر ورودی:           فیلتر 3×3:          نتیجه (Feature Map):
┌─────────────────┐    ┌───────┐           ┌─────────────────┐
│ 1  2  3  4  5  │    │ 1 0 -1│           │ عدد  عدد  ...  │
│ 6  7  8  9  10 │  × │ 2 0 -2│  →  →  → │ ...             │
│ 11 12 13 14 15 │    │ 1 0 -1│           │                 │
│ ...             │    └───────┘           └─────────────────┘
└─────────────────┘
```

**مزایا:**
- فیلتر 3×3 فقط **9 وزن** دارد (نه میلیون‌ها)
- همان فیلتر روی سراسر تصویر استفاده می‌شود (Local Pattern Learning)
- ویژگی‌های مکانی را حفظ می‌کند

### مدل ما: Fully Convolutional Network

```
ورودی (H×W×4)
    ↓
[Conv 3×3 × 32] + BN + ReLU   ← بلاک 1: ویژگی‌های ساده
[Conv 3×3 × 32] + BN + ReLU
    ↓
[Conv 3×3 × 64] + BN + ReLU   ← بلاک 2: ویژگی‌های پیچیده‌تر
[Conv 3×3 × 64] + BN + ReLU
    ↓
[Conv 3×3 × 128] + BN + ReLU  ← بلاک 3: ویژگی‌های انتزاعی
[Conv 3×3 × 128] + BN + ReLU
    ↓
[Conv 1×1 × 6]                 ← لایه خروجی: ۶ کانال (۶ کلاس)
    ↓
Softmax                         ← احتمال هر کلاس
خروجی (H×W×6)
```

**نکته مهم:** هیچ MaxPooling نداریم → اندازه تصویر تغییر نمی‌کند.
هر پیکسل ورودی مستقیماً به یک احتمال کلاس تبدیل می‌شود.


### مفاهیم کلیدی لایه‌ها

#### BatchNormalization چیست؟
در طول آموزش، اعداد می‌توانند خیلی بزرگ یا کوچک شوند.
BatchNorm آن‌ها را **نرمال** نگه می‌دارد (میانگین~۰، واریانس~۱).

نتیجه: آموزش پایدارتر، سریع‌تر، و بهتر.

#### ReLU (Rectified Linear Unit) چیست؟

```
ReLU(x) = max(0, x)
```
- اعداد منفی → صفر می‌شوند
- اعداد مثبت → بدون تغییر

چرا؟ شبکه عصبی باید **غیرخطی** باشد. بدون ReLU، ترکیب هر تعداد لایه
خطی فقط یک تابع خطی می‌شود (بی‌فایده برای مسائل پیچیده).

#### Softmax چیست؟

```
softmax([2.0, 1.0, 0.5]) = [0.66, 0.24, 0.10]
```
مقادیر را به **احتمالات** تبدیل می‌کند (مجموع = ۱).
برای هر پیکسل: احتمال تعلق به هر یک از ۶ کلاس.

#### Conv2D(1×1) چیست؟
یک کانولوشن ۱×۱ مثل یک **خلاصه‌کننده** کانال‌هاست.
128 کانال را به 6 کانال کاهش می‌دهد بدون اثر بر اندازه تصویر.


### ۲.۱ — تابع بارگذاری داده

#### چرا به یک تابع خاص نیاز داریم؟

وقتی dataset بزرگ است (هزاران فایل)، نمی‌توانیم همه را یکجا به RAM بیاوریم.
باید هر دسته (batch) را موقع نیاز بخوانیم.

#### تابع `load_sample` چه کار می‌کند؟
۱. فایل GeoTIFF را باز می‌کند
۲. باندهای ورودی را جدا می‌کند (RGB+IR یا RGB+IR+Elevation)
۳. هر باند را نرمال می‌کند (بازه ۰ تا ۱)
۴. برچسب را به **One-Hot** تبدیل می‌کند

#### One-Hot Encoding چیست؟

به جای عدد کلاس (مثلاً 4 برای ماشین)، یک **بردار** ۶ تایی می‌سازیم:
```
کلاس 4 (ماشین) → [0, 0, 0, 0, 1, 0]
کلاس 2 (درخت)  → [0, 0, 1, 0, 0, 0]
```
چرا؟ تابع خطا (Categorical Cross-Entropy) به این فرمت نیاز دارد.

#### `data.transpose(1,2,0)` چیست؟
rasterio داده را به شکل `(bands, H, W)` برمی‌گرداند.
TensorFlow داده را به شکل `(H, W, bands)` می‌خواهد.
`transpose(1,2,0)` ترتیب ابعاد را عوض می‌کند:
```
(6, 224, 224) → (224, 224, 6)
```


In [ ]:
# ── هایپرپارامترهای مرحله ۲ ─────────────────────────────────────────
BATCH_SIZE = 2       # تعداد نمونه در هر دسته (batch)
                     # با GPU بزرگتر می‌شود (مثلاً 16 یا 32)
EPOCHS_CNN = 20      # تعداد دوره‌های آموزش
LR_CNN     = 1e-3    # نرخ یادگیری (Learning Rate): 0.001
                     # چقدر با هر گام وزن‌ها تغییر کنند

def load_sample(file_path, use_elevation=False):
    """
    یک فایل GeoTIFF را می‌خواند و آماده آموزش می‌کند.
    
    پارامترها:
        file_path     : مسیر فایل .tif
        use_elevation : اگر True → 5 باند (RGB+IR+Elev)
                        اگر False → 4 باند (RGB+IR)
    خروجی:
        X : آرایه ویژگی (H, W, channels)
        y : برچسب One-Hot (H, W, 6)
    """
    with rasterio.open(file_path) as src:
        d = src.read()  # شکل: (6, H, W)
    
    # تعداد باندهای ورودی
    n_bands = 5 if use_elevation else 4
    
    # انتخاب باندها و تغییر ترتیب: (6,H,W) → (H,W,n_bands)
    X = d[:n_bands].transpose(1, 2, 0).astype(np.float32)
    
    # نرمال‌سازی هر باند به بازه [0,1]
    for c in range(X.shape[-1]):
        X[..., c] = normalize_band(X[..., c])
    
    # برچسب One-Hot: (H,W) → (H,W,6)
    label_band = d[5].astype(np.int32)
    y = tf.keras.utils.to_categorical(label_band, num_classes=NUM_CLASSES)
    
    return X, y

# تست تابع با فایل نمونه
X4, y = load_sample(sample_file, use_elevation=False)  # 4 باند
X5, _ = load_sample(sample_file, use_elevation=True)   # 5 باند
print('شکل ورودی 4 باند (RGB+IR)          :', X4.shape)
print('شکل ورودی 5 باند (RGB+IR+Elevation):', X5.shape)
print('شکل برچسب One-Hot                   :', y.shape)
print('\nمثال One-Hot برای پیکسل (0,0):', y[0,0,:])
print('→ کلاس پیکسل:', y[0,0,:].argmax(), '=', CLASS_NAMES[y[0,0,:].argmax()])


### ۲.۲ — تابع Augmentation (افزایش داده)

#### مشکل Overfitting چیست؟

**Overfitting** = مدل داده‌های آموزش را **حفظ** می‌کند ولی **تعمیم** نمی‌دهد.

مثال: دانش‌آموزی که سوال‌های کتاب را حفظ کرده ولی نمی‌تواند تفکر کند.

**علائم Overfitting:**
- Training Accuracy خیلی بالا (مثلاً 99%)
- Validation Accuracy پایین (مثلاً 55%)
- نمودار: Training Loss پایین می‌رود ولی Validation Loss بالا می‌رود

#### Data Augmentation: راه‌حل

با تبدیل‌های تصادفی از هر تصویر **نسخه‌های مختلف** می‌سازیم.
مثل اینکه یک ماشین را از زوایای مختلف بگیریم.

```
تصویر اصلی:    flip افقی:     flip عمودی:    چرخش 90°:
┌─────────┐    ┌─────────┐    ┌─────────┐    ┌─────────┐
│ 🌳 🏠 🚗│    │🚗 🏠 🌳│    │ 🚗 🏠 🌳│    │ 🏠 🚗 🌳│
│ 🛣️  🛣️ │    │ 🛣️  🛣️│    │ 🛣️  🛣️│    │ ...     │
└─────────┘    └─────────┘    └─────────┘    └─────────┘
```

**مهم:** برچسب هم باید همان تبدیل را ببیند! (X و y باید همزمان flip شوند)


In [ ]:
def augment(X, y):
    """
    تبدیل‌های تصادفی روی داده ورودی و برچسب اعمال می‌کند.
    
    مهم: هر تبدیل روی X و y هر دو اعمال می‌شود
    تا ارتباط پیکسل-به-پیکسل حفظ شود.
    
    ورودی: X → (H, W, C)  |  y → (H, W, 6)
    خروجی: همان شکل ولی تصادفاً تبدیل شده
    """
    # flip افقی (آینه از چپ به راست)
    # [:, ::-1, :] یعنی ستون‌ها را معکوس کن
    if np.random.rand() > 0.5:
        X = X[:, ::-1, :]   # معکوس کردن ستون‌ها در X
        y = y[:, ::-1, :]   # همان کار برای y
    
    # flip عمودی (آینه از بالا به پایین)
    # [::-1, :, :] یعنی ردیف‌ها را معکوس کن
    if np.random.rand() > 0.5:
        X = X[::-1, :, :]   # معکوس کردن ردیف‌ها در X
        y = y[::-1, :, :]   # همان کار برای y
    
    # چرخش تصادفی: 0°، 90°، 180° یا 270°
    k = np.random.randint(0, 4)  # عدد تصادفی 0 تا 3
    # np.rot90: ماتریس را k×90 درجه می‌چرخاند
    X = np.rot90(X, k).copy()   # .copy() برای حافظه پیوسته
    y = np.rot90(y, k).copy()
    
    return X, y


def make_dataset(file_list, augment_data=False, batch_size=2, use_elevation=False):
    """
    یک tf.data.Dataset از لیست فایل‌ها می‌سازد.
    
    tf.data.Dataset: یک pipeline کارآمد برای خواندن و پردازش داده.
    به جای بارگذاری همه داده در حافظه، داده را در هنگام نیاز می‌خواند.
    
    متدها:
    .shuffle(): ترتیب نمونه‌ها را تصادفی می‌کند
    .batch(n): n نمونه را گروه‌بندی می‌کند
    .prefetch(): در پس‌زمینه داده بعدی را آماده می‌کند (سریع‌تر)
    """
    Xs, ys = [], []
    for fp in file_list:
        X, y = load_sample(fp, use_elevation=use_elevation)
        if augment_data:
            X, y = augment(X, y)  # تبدیل تصادفی فقط برای training
        Xs.append(X)
        ys.append(y)
    
    # تبدیل لیست آرایه‌ها به یک آرایه بزرگ
    X_arr = np.stack(Xs)  # شکل: (N, H, W, C)
    y_arr = np.stack(ys)  # شکل: (N, H, W, 6)
    
    # ساخت Dataset
    ds = tf.data.Dataset.from_tensor_slices((X_arr, y_arr))
    if augment_data:
        ds = ds.shuffle(len(file_list), seed=SEED)  # ترتیب تصادفی
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


print('توابع بارگذاری داده آماده شدند!')
print('\nساخت Dataset‌ها ...')
train_ds = make_dataset(train_files, augment_data=True,  batch_size=BATCH_SIZE)
val_ds   = make_dataset(val_files,   augment_data=False, batch_size=BATCH_SIZE)
test_ds  = make_dataset(test_files,  augment_data=False, batch_size=BATCH_SIZE)
print('Training dataset   آماده شد')
print('Validation dataset آماده شد')
print('Test dataset       آماده شد')


### ۲.۳ — ساخت معماری مدل CNN

#### `keras.Input` چیست؟
دروازه ورودی مدل. `shape=(None, None, 4)` یعنی:
- `None, None`: هر اندازه‌ای از تصویر (انعطاف‌پذیر)
- `4`: تعداد کانال‌های ورودی (RGB + IR)

#### `layers.Conv2D(32, 3, padding='same')` چیست؟
- `32`: تعداد فیلتر (هر فیلتر یک ویژگی یاد می‌گیرد)
- `3`: اندازه فیلتر (3×3)
- `padding='same'`: لبه‌ها را با صفر پر کن تا اندازه خروجی = ورودی

#### چرا فیلترها بیشتر می‌شوند (32 → 64 → 128)?

لایه‌های اول ویژگی‌های ساده می‌آموزند (لبه، رنگ).
لایه‌های بعدی ویژگی‌های پیچیده‌تر (بافت، شکل، الگوی ساختمان).
برای ویژگی‌های پیچیده‌تر به فیلترهای بیشتری نیاز داریم.

#### `activation='relu'` چیست؟
ReLU به عنوان آرگومان در Conv2D → بعد از کانولوشن بلافاصله ReLU اعمال می‌شود.


In [ ]:
def build_simple_cnn(input_channels=4, num_classes=6):
    """
    مدل Simple CNN برای تقسیم‌بندی معنایی می‌سازد.
    
    معماری: Fully Convolutional Network
    ورودی: (H, W, input_channels)
    خروجی: (H, W, num_classes) — احتمال هر کلاس برای هر پیکسل
    """
    # ── ورودی ────────────────────────────────────────────────────────
    inp = keras.Input(shape=(None, None, input_channels), name='input')
    
    # ── بلاک ۱: فیلترهای ساده (32 فیلتر) ───────────────────────────
    # این بلاک ویژگی‌های پایه مثل لبه و رنگ را یاد می‌گیرد
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)  # پایدارسازی
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    
    # ── بلاک ۲: فیلترهای متوسط (64 فیلتر) ──────────────────────────
    # این بلاک ویژگی‌های پیچیده‌تر مثل بافت سطح را یاد می‌گیرد
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    
    # ── بلاک ۳: فیلترهای عمیق (128 فیلتر) ──────────────────────────
    # این بلاک ویژگی‌های انتزاعی مثل شکل ساختمان را یاد می‌گیرد
    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    
    # ── لایه خروجی ──────────────────────────────────────────────────
    # Conv 1×1: 128 کانال را به 6 کانال (تعداد کلاس) کاهش می‌دهد
    # softmax: احتمالات را محاسبه می‌کند (مجموع = 1 برای هر پیکسل)
    out = layers.Conv2D(num_classes, 1, padding='same',
                        activation='softmax', name='output')(x)
    
    model = keras.Model(inputs=inp, outputs=out, name='SimpleCNN')
    return model


# ساخت مدل
cnn = build_simple_cnn(input_channels=4, num_classes=NUM_CLASSES)

# تنظیم optimizer و تابع خطا
# Adam: بهترین optimizer عمومی برای یادگیری عمیق
# categorical_crossentropy: تابع خطای استاندارد برای طبقه‌بندی چندکلاسه
cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR_CNN),
    loss='categorical_crossentropy',
    metrics=['accuracy']  # دوستانه برای گزارش
)

# خلاصه مدل
print('مشخصات مدل Simple CNN:')
cnn.summary()
print(f'\nمجموع پارامترها: {cnn.count_params():,}')


### ۲.۴ — آموزش مدل

#### فرایند آموزش چگونه کار می‌کند؟

در هر **epoch** (دور):
1. یک **batch** (دسته کوچک) از training data می‌گیریم
2. Forward Pass: داده از ابتدا تا انتهای مدل عبور می‌کند → پیش‌بینی
3. محاسبه خطا: **Loss** = اختلاف بین پیش‌بینی و واقعیت
4. Backward Pass: gradient (شیب) Loss نسبت به وزن‌ها محاسبه می‌شود
5. **Adam Optimizer** وزن‌ها را کمی در جهت کاهش Loss تغییر می‌دهد
6. تکرار برای تمام batch‌های training
7. ارزیابی روی Validation set

#### ModelCheckpoint چیست؟

یک **callback** (عملکرد جانبی) است که بهترین مدل را ذخیره می‌کند.
```python
monitor='val_accuracy'  # معیار ذخیره: val_accuracy
save_best_only=True     # فقط وقتی بهتر می‌شود ذخیره کن
```
چرا؟ چون مدل در epoch آخر بهترین نیست — Overfitting می‌کند.
باید بهترین checkpoint را نگه داریم.

#### Learning Rate (LR) چیست?

مثل اندازه قدم در پیاده‌روی:
- LR خیلی بزرگ → قدم‌های بلند → از مسیر خارج می‌شوی
- LR خیلی کوچک → قدم‌های خیلی کوچک → خیلی کُند
- LR = 0.001 (1e-3) → متوازن


In [ ]:
# ── آموزش مدل CNN ───────────────────────────────────────────────────

# مسیر ذخیره بهترین مدل
best_cnn_path = os.path.join(DATA_DIR, 'best_simple_model.keras')

# ModelCheckpoint: ناظری که بهترین مدل را ذخیره می‌کند
checkpoint_cnn = ModelCheckpoint(
    filepath=best_cnn_path,
    monitor='val_accuracy',    # معیار مقایسه: دقت validation
    save_best_only=True,       # فقط وقتی بهتر می‌شود
    mode='max',                # بزرگتر = بهتر (برای accuracy)
    verbose=0                  # بدون پیام اضافه
)

print('شروع آموزش ...')
print(f'تعداد epoch: {EPOCHS_CNN}')
print(f'نرخ یادگیری: {LR_CNN}')
print(f'Batch size: {BATCH_SIZE}')
print('=' * 50)

# model.fit: تابع اصلی آموزش
history_cnn = cnn.fit(
    train_ds,                     # داده‌های آموزش
    validation_data=val_ds,        # داده‌های اعتبارسنجی (برای نظارت)
    epochs=EPOCHS_CNN,             # تعداد دوره
    callbacks=[checkpoint_cnn],    # ناظرها
    verbose=1                      # نمایش پیشرفت
)

print('\nآموزش تمام شد!')
print(f'بهترین val_accuracy: {max(history_cnn.history["val_accuracy"])*100:.2f}%')


### ۲.۵ — نمودار آموزش و تحلیل

#### چگونه نمودار را تفسیر کنیم؟

**حالت ایده‌آل:**
```
Loss                     Accuracy
  ↓ Training              Training ↑
  ↓ Validation            Validation ↑
  [هر دو نزدیک هم]       [هر دو نزدیک هم]
```

**حالت Overfitting:**
```
Loss                     Accuracy
  ↓ Training              Training ↑↑↑ (خیلی بالا)
  ↑ Validation            Validation → (ثابت یا پایین)
  [فاصله زیاد!]           [فاصله زیاد!]
```

با **dataset کوچک** (یک فایل تکرار شده) انتظار Overfitting داریم.
این کاملاً طبیعی است و با dataset کامل بهتر می‌شود.


In [ ]:
# ── رسم نمودار آموزش ────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, EPOCHS_CNN + 1)

# نمودار Loss
ax = axes[0]
ax.plot(epochs_range, history_cnn.history['loss'],
        color='royalblue', linewidth=2, label='Training Loss')
ax.plot(epochs_range, history_cnn.history['val_loss'],
        color='darkorange', linewidth=2, linestyle='--', label='Validation Loss')
ax.set_title('Simple CNN — مقایسه Loss', fontsize=12, fontweight='bold')
ax.set_xlabel('Epoch (دوره)')
ax.set_ylabel('Loss (خطا)')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# نمودار Accuracy
ax = axes[1]
ax.plot(epochs_range, [v*100 for v in history_cnn.history['accuracy']],
        color='forestgreen', linewidth=2, label='Training Acc')
ax.plot(epochs_range, [v*100 for v in history_cnn.history['val_accuracy']],
        color='crimson', linewidth=2, linestyle='--', label='Validation Acc')
ax.set_title('Simple CNN — مقایسه Accuracy', fontsize=12, fontweight='bold')
ax.set_xlabel('Epoch (دوره)')
ax.set_ylabel('Accuracy (%)')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
curve_path = os.path.join(DATA_DIR, 'step2_training_curves.png')
plt.savefig(curve_path, dpi=150)
plt.close()
display(Image(curve_path))

# تحلیل خودکار
final_train_acc = history_cnn.history['accuracy'][-1] * 100
final_val_acc   = history_cnn.history['val_accuracy'][-1] * 100
gap = final_train_acc - final_val_acc
print(f'Train Accuracy:      {final_train_acc:.2f}%')
print(f'Validation Accuracy: {final_val_acc:.2f}%')
print(f'فاصله (Gap):        {gap:.2f}%')
if gap > 10:
    print('  → Overfitting تشخیص داده شد (فاصله > 10%)')
    print('  → با dataset کامل این مشکل کمتر خواهد شد')


### ۲.۶ — ارزیابی نهایی روی Test Set

#### چرا Test Set جداست؟

Test Set را **هرگز** در آموزش یا انتخاب مدل استفاده نکردیم.
ارزیابی روی Test Set = **آزمون واقعی** که نشان می‌دهد مدل چقدر به دنیای واقعی تعمیم می‌دهد.

#### چرا بهترین مدل را بارگذاری می‌کنیم؟
مدل در epoch آخر لزوماً بهترین نیست.
ModelCheckpoint بهترین epoch را ذخیره کرده — آن را بارگذاری می‌کنیم.


In [ ]:
# بارگذاری بهترین مدل ذخیره شده
best_cnn = keras.models.load_model(best_cnn_path)
print('بهترین مدل CNN بارگذاری شد')

# ارزیابی روی test set
test_loss_cnn, test_acc_cnn = best_cnn.evaluate(test_ds, verbose=0)

print('\n' + '=' * 45)
print('   نتایج Simple CNN روی Test Set   ')
print('=' * 45)
print(f'  Test Loss     : {test_loss_cnn:.4f}')
print(f'  Test Accuracy : {test_acc_cnn*100:.2f}%')
print('=' * 45)
print()
print('  Train Accuracy (آخرین epoch):', f'{history_cnn.history["accuracy"][-1]*100:.2f}%')
print('  Test Accuracy               :', f'{test_acc_cnn*100:.2f}%')
overfit_gap = history_cnn.history['accuracy'][-1]*100 - test_acc_cnn*100
print(f'  فاصله Overfitting           : {overfit_gap:.2f}%')


### خلاصه مرحله ۲

**مفاهیمی که یاد گرفتیم:**
- کانولوشن چیست و چرا از شبکه کاملاً متصل بهتر است
- BatchNormalization، ReLU، Softmax هر کدام چه نقشی دارند
- Overfitting چیست و چطور آن را از نمودار می‌شناسیم
- K-Fold چرا ارزیابی عادلانه‌تری نسبت به Train/Test ساده می‌دهد

**نتیجه:**
مدل CNN ساده Overfitting می‌کند چون dataset کوچک است.
در مرحله ۳ با معماری **U-Net** این مشکل را بهتر مدیریت می‌کنیم.

**→ ادامه: U-Net — معماری حرفه‌ای برای تقسیم‌بندی تصویر**


---
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# مرحله ۳: مدل U-Net (Encoder-Decoder با Skip Connections)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## مشکل اصلی CNN ساده

در مرحله ۲، مدل CNN ساده داشتیم که اندازه تصویر ثابت ماند.
خوب بود، ولی یک ضعف اساسی دارد:

**جزئیات مکانی ظریف را یاد نمی‌گیرد.**

برای تشخیص دقیق مرز بین ساختمان و آسفالت — به جزئیات ظریف نیاز داریم.

## راه‌حل: U-Net

U-Net در سال ۲۰۱۵ برای تصویرسازی پزشکی ساخته شد.
امروز در بینایی ماهواره‌ای هم استاندارد است.

### ایده اصلی: Encoder-Decoder با Skip Connection

```
[ورودی: تصویر کامل]
     |
  [Encoder: فشرده‌سازی]
     |
     ↓ MaxPool → تصویر نصف می‌شود (اما ویژگی بیشتر)
     ↓ MaxPool → تصویر ربع می‌شود
     ↓ MaxPool → ...
     |
  [Bottleneck: فشرده‌ترین نقطه — اطلاعات انتزاعی]
     |
  [Decoder: بازسازی]
     ↑ UpSample → تصویر دو برابر می‌شود
     ↑ UpSample → ...
     |
[خروجی: نقشه کلاس با همان اندازه ورودی]
```

### Skip Connection: نوآوری اصلی U-Net

```
Encoder       Bottleneck    Decoder
────────                    ────────
Block1 ──────────────────→ Concat → Block1'
  ↓
Block2 ────────────────→ Concat → Block2'
  ↓
Block3 ──────────────→ Concat → Block3'
  ↓
Block4 ────────────→ Concat → Block4'
  ↓
Bottleneck ─────────────────────────────→
```

**Skip Connection = پل مستقیم** از عمق encoder به decoder.
- اطلاعات مکانی دقیق (لبه‌ها، مرزها) از Block1 مستقیم به Decoder می‌رود
- Decoder برای بازسازی به این اطلاعات نیاز دارد

**چرا بهتر است؟**
- CNN ساده: ویژگی‌های ریز (لبه ساختمان) در لایه‌های عمیق گم می‌شوند
- U-Net: این ویژگی‌های ریز مستقیم به Decoder منتقل می‌شوند


### ۳.۱ — مفاهیم جدید در U-Net

#### MaxPooling چیست؟

تصویر را نصف می‌کند — ولی ویژگی‌های مهم را نگه می‌دارد:
```
قبل از MaxPool:      بعد از MaxPool 2×2:
┌──┬──┬──┬──┐         ┌──┬──┐
│1 │3 │2 │4 │  →→→    │3 │4 │   (بیشترین ارزش در هر 2×2)
│5 │7 │6 │8 │         │7 │9 │
├──┼──┼──┼──┤
│2 │4 │9 │1 │
│6 │8 │3 │2 │
└──┴──┴──┴──┘
```
اندازه نصف می‌شود → سریع‌تر → میدان دید بزرگتر (global context)

#### UpSampling (آپ‌سمپلینگ) چیست؟

عکس MaxPool — تصویر را دو برابر می‌کند:
```
┌──┬──┐         ┌──┬──┬──┬──┐
│3 │4 │  →→→    │3 │3 │4 │4 │
│7 │9 │         │7 │7 │9 │9 │
└──┴──┘         └──┴──┴──┴──┘
```
ساده‌ترین روش: هر پیکسل را ۴ بار تکرار کن (Nearest Neighbor Upsampling)

#### Concatenate (ترکیب) چیست؟

وقتی Feature Map از UpSampling با Skip Connection ترکیب می‌شود:
```
از UpSampling: (H, W, 256)
از Skip Conn:  (H, W, 256)
بعد از Concat: (H, W, 512)  ← کانال‌ها کنار هم قرار می‌گیرند
```
Decoder حالا هم اطلاعات انتزاعی (از bottleneck) و هم اطلاعات ظریف (از encoder) دارد.


In [ ]:
# ── ساخت بلاک‌های U-Net ────────────────────────────────────────────

from tensorflow.keras import layers

def conv_block(x, num_filters, block_name):
    """
    یک بلاک کانولوشنی می‌سازد: Conv → BN → ReLU → Conv → BN → ReLU
    
    این پایه‌ای‌ترین بلاک است که هم در Encoder و هم Decoder استفاده می‌شود.
    
    پارامترها:
        x           : تنسور ورودی
        num_filters : تعداد فیلترهای کانولوشن
        block_name  : نام این بلاک در گراف مدل
    خروجی: تنسور با همان H,W ولی num_filters کانال
    """
    x = layers.Conv2D(
        num_filters, 3,             # 3×3 kernel
        padding='same',             # اندازه حفظ می‌شود
        activation='relu',          # فعال‌سازی غیرخطی
        name=f'{block_name}_c1'    # نام‌گذاری برای ردیابی
    )(x)
    x = layers.BatchNormalization(name=f'{block_name}_bn1')(x)
    x = layers.Conv2D(
        num_filters, 3,
        padding='same',
        activation='relu',
        name=f'{block_name}_c2'
    )(x)
    x = layers.BatchNormalization(name=f'{block_name}_bn2')(x)
    return x


def build_unet(input_channels=5, num_classes=6, base_filters=32):
    """
    معماری کامل U-Net برای تقسیم‌بندی معنایی.
    
    پارامترها:
        input_channels: تعداد کانال‌های ورودی (5 = RGB+IR+Elevation)
        num_classes   : تعداد کلاس‌های خروجی (6)
        base_filters  : تعداد پایه فیلترها (32 → 64 → 128 → 256 → 512)
    """
    f = base_filters  # مخفف برای خوانایی
    
    inp = keras.Input(shape=(None, None, input_channels), name='input')
    
    # ────────────── ENCODER ──────────────────────────────────────────
    # هر مرحله: conv_block → ذخیره skip → MaxPool (نصف کردن)
    
    e1 = conv_block(inp, f,   'enc1')   # (H,  W,  32)
    p1 = layers.MaxPooling2D(2, name='pool1')(e1)  # (H/2, W/2, 32)
    
    e2 = conv_block(p1,  f*2, 'enc2')   # (H/2, W/2, 64)
    p2 = layers.MaxPooling2D(2, name='pool2')(e2)  # (H/4, W/4, 64)
    
    e3 = conv_block(p2,  f*4, 'enc3')   # (H/4,  W/4,  128)
    p3 = layers.MaxPooling2D(2, name='pool3')(e3)  # (H/8, W/8, 128)
    
    e4 = conv_block(p3,  f*8, 'enc4')   # (H/8,  W/8,  256)
    p4 = layers.MaxPooling2D(2, name='pool4')(e4)  # (H/16, W/16, 256)
    
    # ────────────── BOTTLENECK ───────────────────────────────────────
    # فشرده‌ترین نقطه — انتزاعی‌ترین اطلاعات
    b = conv_block(p4, f*16, 'bottleneck')   # (H/16, W/16, 512)
    
    # ────────────── DECODER ──────────────────────────────────────────
    # هر مرحله: UpSampling → Concatenate با skip → conv_block
    
    # مرحله ۱ Decoder:
    u4 = layers.UpSampling2D(2, name='up4')(b)           # (H/8,  W/8,  512)
    u4 = layers.Concatenate(name='cat4')([u4, e4])       # ← Skip از enc4
    d4 = conv_block(u4, f*8, 'dec4')                     # (H/8,  W/8,  256)
    
    # مرحله ۲ Decoder:
    u3 = layers.UpSampling2D(2, name='up3')(d4)          # (H/4,  W/4,  256)
    u3 = layers.Concatenate(name='cat3')([u3, e3])       # ← Skip از enc3
    d3 = conv_block(u3, f*4, 'dec3')                     # (H/4,  W/4,  128)
    
    # مرحله ۳ Decoder:
    u2 = layers.UpSampling2D(2, name='up2')(d3)          # (H/2,  W/2,  128)
    u2 = layers.Concatenate(name='cat2')([u2, e2])       # ← Skip از enc2
    d2 = conv_block(u2, f*2, 'dec2')                     # (H/2,  W/2,  64)
    
    # مرحله ۴ Decoder:
    u1 = layers.UpSampling2D(2, name='up1')(d2)          # (H,    W,    64)
    u1 = layers.Concatenate(name='cat1')([u1, e1])       # ← Skip از enc1
    d1 = conv_block(u1, f, 'dec1')                       # (H,    W,    32)
    
    # ────────────── OUTPUT ───────────────────────────────────────────
    # Conv 1×1: 32 → 6 کانال (یک کانال برای هر کلاس)
    out = layers.Conv2D(num_classes, 1,
                        padding='same', activation='softmax',
                        name='output')(d1)  # (H, W, 6)
    
    model = keras.Model(inputs=inp, outputs=out, name='UNet')
    return model


# ساخت U-Net
unet = build_unet(input_channels=5, num_classes=NUM_CLASSES, base_filters=32)
unet.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),  # LR کمتر از CNN ساده
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('مشخصات مدل U-Net:')
unet.summary()
print(f'\nمجموع پارامترها: {unet.count_params():,}')
print(f'در مقایسه با CNN ساده (215K): U-Net {unet.count_params()/215_000:.1f}x بزرگتر است')


### ۳.۲ — چرا Learning Rate کمتری برای U-Net؟

U-Net بزرگتر است (~۷.۸۶ میلیون پارامتر در برابر ~۲۱۵ هزار).

با مدل‌های بزرگتر:
- گام‌های کوچکتر (LR پایین‌تر) → آموزش پایدارتر
- `1e-4` = 0.0001 (ده برابر کمتر از CNN ساده که `1e-3` بود)

### ۳.۳ — آموزش U-Net

در این بخش از **۵ باند** استفاده می‌کنیم (RGB + IR + Elevation).
چرا ارتفاع مهم است؟
- ساختمان: ارتفاع بالا
- ماشین: ارتفاع کم اما مثبت
- آسفالت: ارتفاع نزدیک صفر


In [ ]:
# ── ساخت Dataset‌های U-Net با 5 باند ───────────────────────────────

EPOCHS_UNET = 20

print('ساخت Dataset‌ها با 5 باند (RGB+IR+Elevation) ...')
train_ds_u = make_dataset(train_files, augment_data=True,  batch_size=BATCH_SIZE, use_elevation=True)
val_ds_u   = make_dataset(val_files,   augment_data=False, batch_size=BATCH_SIZE, use_elevation=True)
test_ds_u  = make_dataset(test_files,  augment_data=False, batch_size=BATCH_SIZE, use_elevation=True)

# Checkpoint برای بهترین U-Net
best_unet_path = os.path.join(DATA_DIR, 'best_unet_model.keras')
checkpoint_u = ModelCheckpoint(
    filepath=best_unet_path,
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=0
)

print(f'شروع آموزش U-Net برای {EPOCHS_UNET} epoch ...')
print('(این ممکن است چند دقیقه طول بکشد)')
print('=' * 50)

history_unet = unet.fit(
    train_ds_u,
    validation_data=val_ds_u,
    epochs=EPOCHS_UNET,
    callbacks=[checkpoint_u],
    verbose=1
)

print('\nآموزش U-Net تمام شد!')
print(f'بهترین val_accuracy: {max(history_unet.history["val_accuracy"])*100:.2f}%')


In [ ]:
# ── مقایسه نمودار آموزش CNN ساده در برابر U-Net ──────────────────────

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('مقایسه Simple CNN در برابر U-Net', fontsize=14, fontweight='bold')

ep_cnn  = range(1, EPOCHS_CNN+1)
ep_unet = range(1, EPOCHS_UNET+1)

# ردیف ۱: Simple CNN
axes[0,0].plot(ep_cnn, history_cnn.history['loss'],     label='Train', color='royalblue', lw=2)
axes[0,0].plot(ep_cnn, history_cnn.history['val_loss'], label='Val',   color='darkorange', lw=2, ls='--')
axes[0,0].set_title('Simple CNN — Loss', fontweight='bold')
axes[0,0].set_xlabel('Epoch'); axes[0,0].set_ylabel('Loss')
axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

axes[0,1].plot(ep_cnn, [v*100 for v in history_cnn.history['accuracy']],     label='Train', color='forestgreen', lw=2)
axes[0,1].plot(ep_cnn, [v*100 for v in history_cnn.history['val_accuracy']], label='Val',   color='crimson', lw=2, ls='--')
axes[0,1].set_title('Simple CNN — Accuracy', fontweight='bold')
axes[0,1].set_xlabel('Epoch'); axes[0,1].set_ylabel('Accuracy (%)')
axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

# ردیف ۲: U-Net
axes[1,0].plot(ep_unet, history_unet.history['loss'],     label='Train', color='royalblue', lw=2)
axes[1,0].plot(ep_unet, history_unet.history['val_loss'], label='Val',   color='darkorange', lw=2, ls='--')
axes[1,0].set_title('U-Net — Loss', fontweight='bold')
axes[1,0].set_xlabel('Epoch'); axes[1,0].set_ylabel('Loss')
axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

axes[1,1].plot(ep_unet, [v*100 for v in history_unet.history['accuracy']],     label='Train', color='forestgreen', lw=2)
axes[1,1].plot(ep_unet, [v*100 for v in history_unet.history['val_accuracy']], label='Val',   color='crimson', lw=2, ls='--')
axes[1,1].set_title('U-Net — Accuracy', fontweight='bold')
axes[1,1].set_xlabel('Epoch'); axes[1,1].set_ylabel('Accuracy (%)')
axes[1,1].legend(); axes[1,1].grid(alpha=0.3)

plt.tight_layout()
compare_path = os.path.join(DATA_DIR, 'step3_training_curves.png')
plt.savefig(compare_path, dpi=150)
plt.close()
display(Image(compare_path))


### ۳.۴ — تصویرسازی پیش‌بینی U-Net

این مهم‌ترین بخش از دیدگاه یادگیری است.

**Argmax چیست؟**
خروجی U-Net برای هر پیکسل: `[0.05, 0.01, 0.02, 0.10, 0.80, 0.02]` (احتمالات 6 کلاس)
`argmax` → شاخص بزرگترین عدد = `4` (ماشین)

```python
pred_label = np.argmax(pred_logit, axis=-1)
# (H, W, 6) → (H, W)   هر پیکسل: کلاس محتمل‌ترین
```


In [ ]:
# ── بارگذاری بهترین U-Net و ارزیابی ────────────────────────────────

best_unet = keras.models.load_model(best_unet_path)
test_loss_u, test_acc_u = best_unet.evaluate(test_ds_u, verbose=0)

print('نتایج U-Net روی Test Set:')
print(f'  Test Loss    : {test_loss_u:.4f}')
print(f'  Test Accuracy: {test_acc_u*100:.2f}%')

# ── پیش‌بینی روی یک تایل ─────────────────────────────────────────────

test_fp = test_files[0]
X5_test, _ = load_sample(test_fp, use_elevation=True)

# پیش‌بینی: افزودن بُعد batch → (1,H,W,5)
X5_batch = np.expand_dims(X5_test, axis=0)  # شکل: (1, H, W, 5)
pred_probs = best_unet.predict(X5_batch, verbose=0)[0]  # شکل: (H, W, 6)

# argmax: بزرگترین احتمال = کلاس پیش‌بینی‌شده
pred_label = np.argmax(pred_probs, axis=-1)  # شکل: (H, W)

# داده اصلی برای نمایش
with rasterio.open(test_fp) as src:
    raw = src.read()

rgb_v    = np.stack([normalize_band(raw[0].astype(np.float32)),
                     normalize_band(raw[1].astype(np.float32)),
                     normalize_band(raw[2].astype(np.float32))], axis=-1)
elev_v   = normalize_band(raw[4].astype(np.float32))
gt_label = raw[5].astype(np.int32)
gt_rgb   = label_to_rgb(gt_label,   CLASS_COLORS)
pred_rgb = label_to_rgb(pred_label, CLASS_COLORS)

patches = [mpatches.Patch(color=[c/255 for c in CLASS_COLORS[i]], label=CLASS_NAMES[i])
           for i in range(NUM_CLASSES)]

# رسم 4 پنجره مقایسه
fig, axes = plt.subplots(1, 4, figsize=(22, 6))
fig.suptitle('U-Net: پیش‌بینی در برابر واقعیت (Ground Truth)', fontsize=13, fontweight='bold')

axes[0].imshow(rgb_v);    axes[0].set_title('تصویر RGB ورودی');   axes[0].axis('off')
im = axes[1].imshow(elev_v, cmap='terrain')
axes[1].set_title('باند ارتفاع'); axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
axes[2].imshow(gt_rgb);   axes[2].set_title('برچسب واقعی (GT)'); axes[2].axis('off')
axes[2].legend(handles=patches, loc='lower right', fontsize=6.5, framealpha=0.9)
axes[3].imshow(pred_rgb); axes[3].set_title('پیش‌بینی U-Net');   axes[3].axis('off')
axes[3].legend(handles=patches, loc='lower right', fontsize=6.5, framealpha=0.9)

plt.tight_layout()
pred_path = os.path.join(DATA_DIR, 'step3_prediction_visualization.png')
plt.savefig(pred_path, dpi=150, bbox_inches='tight')
plt.close()
display(Image(pred_path))
print('تصویر پیش‌بینی ذخیره شد')


---
# 📊 مقایسه نهایی و نتیجه‌گیری

## جدول مقایسه دو مدل

| معیار | Simple CNN | U-Net |
|---|---|---|
| **باندهای ورودی** | ۴ (RGB+IR) | ۵ (RGB+IR+Elevation) |
| **پارامترها** | ~۲۱۵,۰۰۰ | ~۷,۸۶۰,۰۰۰ |
| **زمان آموزش** | سریع‌تر | کندتر |
| **معماری** | Flat | Encoder-Decoder |
| **Skip Connection** | ندارد | دارد |
| **جزئیات مکانی** | ضعیف | قوی |

## چرا U-Net با dataset کامل بهتر خواهد بود؟

با یک فایل تکراری، هر دو مدل Overfit می‌کنند و مشابه به نظر می‌رسند.
وقتی dataset کامل داریم (هزاران تایل متنوع):
- Skip connections اهمیت واقعی خود را نشان می‌دهند
- U-Net مرزهای ظریف بین کلاس‌ها را بهتر پیدا می‌کند
- انتظار: **U-Net بیش از ۸۰٪ دقت** روی Test Set واقعی


In [ ]:
# ── جدول مقایسه نهایی ────────────────────────────────────────────────

print('\n' + '=' * 60)
print('   جدول مقایسه نهایی مدل‌ها')
print('=' * 60)
print(f'{"مدل":<15}{"باند":<12}{"Test Loss":>12}{"Test Acc":>12}')
print('-' * 53)
print(f'{"Simple CNN":<15}{"4 (RGB+IR)":<12}{test_loss_cnn:>12.4f}{test_acc_cnn*100:>11.2f}%')
print(f'{"U-Net":<15}{"5 (+Elev)":<12}{test_loss_u:>12.4f}{test_acc_u*100:>11.2f}%')
print('=' * 60)

winner = 'U-Net' if test_loss_u < test_loss_cnn else 'Simple CNN'
improvement = abs(test_loss_cnn - test_loss_u)
print(f'\nمدل برتر (Test Loss کمتر): {winner}')
print(f'بهبود در Loss: {improvement:.4f}')


---
## جمع‌بندی کلی — آنچه یاد گرفتیم

### مفاهیم بنیادی:
| مفهوم | توضیح کوتاه |
|---|---|
| **Semantic Segmentation** | برچسب‌گذاری پیکسل‌به‌پیکسل تصویر |
| **GeoTIFF** | تصویر چندباندی با اطلاعات جغرافیایی |
| **Convolution** | فیلتر کوچک که روی تصویر می‌لغزد — ویژگی‌ها را استخراج می‌کند |
| **BatchNormalization** | پایدارسازی مقادیر در حین آموزش |
| **ReLU** | غیرخطی‌سازی: `max(0,x)` |
| **Softmax** | تبدیل اعداد به احتمالات |
| **K-Fold CV** | ارزیابی عادلانه با تقسیم داده به K بخش |
| **Overfitting** | حفظ داده به جای یادگیری تعمیم‌پذیر |
| **Data Augmentation** | افزایش تنوع داده برای کاهش Overfitting |
| **U-Net** | Encoder-Decoder با Skip Connection برای تقسیم‌بندی دقیق |
| **Skip Connection** | پل مستقیم از Encoder به Decoder — حفظ جزئیات مکانی |

### گام بعدی با Dataset کامل:

1. `DATA_DIR` را به مسیر dataset کامل Potsdam تغییر بده
2. notebook را از ابتدا اجرا کن
3. انتظار داریم هر دو مدل `> 80%` دقت بگیرند
4. U-Net بهتر از Simple CNN خواهد بود

---
*این نوت‌بوک یک پروژه علمی کامل بود — از صفر تا ارزیابی مدل‌های حرفه‌ای.*
*موفق باشی!* 🎓
